## Plot derivative of a given FITS data record

Using a given kernel as [-1, -1, 1, 1] and a derivative type (on-sided, plus-one, plus-two)

## Import routines

In [ ]:
import matplotlib.pyplot as plt
from astropy.io import fits
import numpy as np
%matplotlib widget

## Read input parameters

In [ ]:
# parameters
plot_time = True  # if True, plot using time (s) as x-axis, else use samples
plot_derivative = True  # if True, plot derivative on secondary y-axis
plot_derivative_filename = "dds_plot.png"  # if not None, save derivative plot to this file
kernel_coeficients = [1, 1, -1, -1]  # 1st derivative kernel
kernel_dds_coeficients = [1, 1, -1.5, -2, 0, 1, 0.5]  # 1st derivative kernel
start_sample_for_kernel0 = 0 #[0, -1, -2 , -3] # one-sided kernel
start_sample_for_kernel1 = 1 #[1, 0, -1 , -2] # #CEA/Saclay

if plot_derivative:
    data_dir = "../../ERESOL/CEASaclay/July2025_v5_v20250621_offsetWindow/sep126sam_200p"
    records_file = f"{data_dir}/sep126sam_200p_7.0keV_0.2keV_50x30.fits"
    #data_dir = "../../ERESOL/CEASaclay/July2025_v5_v20250621_offsetWindow/sep317sam_200p"
    #records_file = f"{data_dir}/sep317sam_200p_3.0keV_6.0keV_50x30.fits"
else:
    data_dir = "../../ERESOL/detectionCube/sep2000sam_200p/E1_6keV"
    records_file = f"{data_dir}/sep2000sam_200p_6keV_6keV_50x30.fits"

record_id = 0  # which record to plot

## Read data record in FITS file

In [ ]:
with fits.open(records_file) as hdulist:
    # read TCLOCK keyword from HDU with EXTNAME='GEOCHANNELPARAM'
    tclock = hdulist["GEOCHANNELPARAM"].header["TCLOCK"]
    sampling_rate = 1.0 / tclock  # Hz
    records = hdulist["TESRECORDS"].data
    record_stream = records["ADC"][record_id]
    # read start time of the record
    record_start_time = records["TIME"][record_id]


In [ ]:
if plot_derivative:
    # calculate derivative of record stream using the kernel
    derivate_record_K0 = np.zeros_like(record_stream, dtype=float)
    derivate_record_K1 = np.zeros_like(record_stream, dtype=float)
    derivate_record_dds0 = np.zeros_like(record_stream, dtype=float)
    derivate_record_dds1 = np.zeros_like(record_stream, dtype=float)

    for i in range(1, len(record_stream)-2):
        derivate_record_K0[i] = sum(kernel_coeficients[k] * record_stream[i+start_sample_for_kernel0-k] for k in range(len(kernel_coeficients)))
        derivate_record_K1[i] = sum(kernel_coeficients[k] * record_stream[i+start_sample_for_kernel1-k] for k in range(len(kernel_coeficients)))

    for i in range(3, len(record_stream)-6):
        derivate_record_dds0[i] = sum(kernel_dds_coeficients[k] * record_stream[i+start_sample_for_kernel0-k] for k in range(len(kernel_dds_coeficients)))
        derivate_record_dds1[i] = sum(kernel_dds_coeficients[k] * record_stream[i+start_sample_for_kernel1-k] for k in range(len(kernel_dds_coeficients)))


## Plot record with pulses

In [ ]:
mksize = 3
#mksize = 1
if plot_time:
    x_label = "Time (s)"
    x_data = np.arange(len(record_stream)) / sampling_rate
    x_start_pulse1 = 3500 / sampling_rate
    x_start_pulse2 = 3626 / sampling_rate
    x_limits = (0.0265, 0.029)
    #x_limits = (0.022, 0.042)
    #x_limits = (0.01, 0.08)
else:
    x_label = "Samples"
    x_data = np.arange(len(record_stream))
    x_start_pulse1 = 3500
    x_start_pulse2 = 3626
    x_limits = (3460, 3800)

fig,ax = plt.subplots(figsize=(10,6))

# vertical line to mark the pulses starts
#ax.axvline(x=x_start_pulse1, color='gray', linestyle='--')
#ax.axvline(x=x_start_pulse2, color='gray', linestyle='--')
# plot record_stream 
signal_label = "Original signal"
ax.plot(x_data, record_stream, marker="o", markersize=mksize, label=signal_label) 
ax.set_xlim(x_limits)
ax.set_xlabel(x_label)
ax.set_ylabel("ADC (A.U.)")
ax.legend(loc='upper center')
#ax.legend(loc='upper left')


if plot_derivative:
    # plot derivated_record and derivated_record_dds on a secondary y-axis
    ax2 = ax.twinx()
    ax2.set_ylim(-20, 20)
    # plot standard derivative signal with open circles
    #ax2.plot(x_data, derivate_record_K0, label="Derivative signal", color='C1',marker="o", markersize=mksize) 
    ax2.plot(x_data, derivate_record_K0, label="Derivative signal", 
            marker="o", markerfacecolor='none', markeredgecolor='C1', markersize=mksize, color='C1', ls=':')  # for ADASS
    
    #ax2.plot(x_data, derivate_record_dds0, label="Derivative signal (DDS)", color='C2',marker="o", markersize=mksize)
    ax2.plot(x_data, derivate_record_dds0, label="Derivative signal (DDS)", color='C2',marker="x", markersize=mksize) # for ADASS
    
    #ax2.plot(x_data, derivate_record_dds1, label="Derivative signal (DDS1)", color='C4',marker="o", markersize=mksize)
    # add threshold line at y=6
    ax2.axhline(y=6, color='black', linestyle=':', label='Derivative Threshold') # threshold line
    ax2.axhline(y=0, color='gray', linestyle='--') # zero line
    ax2.set_ylabel("Derivative")
    ax2.legend(loc='upper right')
fig.tight_layout()


In [ ]:
# save figure
if plot_derivative:
    plot_file = f"/home/ceballos/Documents/presentaciones/ADASS2025/{plot_derivative_filename}"
else:
    plot_file = "/home/ceballos/Documents/presentaciones/ADASS2025/pulses_plot.png"
# save with transparent background
fig.savefig(plot_file, dpi=300, transparent=False)